# EuroCropsML x OLMoEarth Benchmark

Evaluate whether OLMoEarth embeddings improve crop classification on EuroCropsML compared to classical remote sensing features.

**Experiments:**
- A: Baseline Features + LogisticRegression
- B: Baseline Features + LightGBM
- C: OLMoEarth Embeddings + LogisticRegression
- D: OLMoEarth Embeddings + LightGBM
- E: Hybrid (Features + Embeddings) + LightGBM

---
## Section 0 — Environment Setup

In [ ]:
# Uncomment for Colab
# !pip install -r requirements.txt umap-learn -q

In [ ]:
import sys, os, time, gc, json, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from src.data.inspect import get_top_classes, generate_class_distribution, count_classes
from src.data.streaming import stream_from_split
from src.data.features import combined_baseline_features
from src.encoder.olmoearth import OLMoEarthEncoder
from src.models.classical import get_classifier
from src.evaluate.metrics import compute_metrics, save_metrics
from src.viz.umap_viz import plot_umap
from src.viz.confusion import plot_confusion_matrix

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

import psutil, platform
print(f"Python: {sys.version}")
print(f"RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB")
print(f"GPU: {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'}")
print(f"Device: {DEVICE}")

In [ ]:
# Paths
DATA_DIR = "E:/RSGIS/OlmoEarth/Project/Data/preprocess/preprocess"
SPLIT_DIR = "E:/RSGIS/OlmoEarth/Project/Data/split/split"
WEIGHTS_PATH = "E:/RSGIS/OlmoEarth/Project/Models/V1.1_Nano/weights.pth"
USE_CASE = "overlap_latvia_vs_estonia"
OUT_DIR = "results"

for d in ["dataset_stats", "features", "embeddings", "figures"]:
    os.makedirs(f"{OUT_DIR}/{d}", exist_ok=True)

---
## Section 1 — Dataset Loading

Three benchmark settings: Top-10, Top-20, Top-50 classes.

In [ ]:
CLASS_SETTINGS = {"small": 10, "medium": 20, "large": 50}

def load_split_data(top_n_classes, batch_size=256):
    """Load train/val/test splits filtered to top-N classes."""
    top_classes = get_top_classes(DATA_DIR, top_n_classes)
    class_filter = set(top_classes)
    label_encoder = LabelEncoder()
    label_encoder.fit(top_classes)

    splits = {}
    for split_key in ["train", "val", "test"]:
        X_list, y_list = [], []
        for _, data, cls_label in stream_from_split(
            DATA_DIR, SPLIT_DIR, USE_CASE,
            split_name="all", split_key=split_key,
            class_filter=class_filter
        ):
            X_list.append(data)
            y_list.append(cls_label)
        if X_list:
            max_T = max(x.shape[0] for x in X_list)
            C = X_list[0].shape[1]
            X = np.zeros((len(X_list), max_T, C), dtype=np.float32)
            for i, x in enumerate(X_list):
                T = min(x.shape[0], max_T)
                X[i, :T, :] = x[:T, :]
            y = label_encoder.transform(y_list)
        else:
            X, y = np.array([]), np.array([])
        splits[split_key] = (X, y)
        del X_list, y_list
        gc.collect()
    return splits, top_classes, label_encoder

In [ ]:
# Load medium setting (20 classes) as primary benchmark
PRIMARY_N = 20
print(f"Loading Top-{PRIMARY_N} classes...")
t0 = time.time()
splits_20, classes_20, le_20 = load_split_data(PRIMARY_N)
X_train_20, y_train_20 = splits_20["train"]
X_val_20, y_val_20 = splits_20["val"]
X_test_20, y_test_20 = splits_20["test"]
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"Train: {len(X_train_20)} | Val: {len(X_val_20)} | Test: {len(X_test_20)}")
print(f"Shape: {X_train_20.shape}")
print(f"Classes: {len(classes_20)}")

---
## Section 2 — Dataset Statistics

In [ ]:
dist = generate_class_distribution(DATA_DIR, top_n=50)
top20_info = dist["top_n_classes"][:20]

fig, ax = plt.subplots(figsize=(12, 5))
classes_names = [c["class"] for c in top20_info]
counts = [c["count"] for c in top20_info]
ax.bar(range(len(counts)), counts, color="steelblue")
ax.set_xticks(range(len(counts)))
ax.set_xticklabels(classes_names, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Samples")
ax.set_title(f"Top-20 Class Distribution (Total: {dist['total_samples']} samples, {dist['total_classes']} classes)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/dataset_stats/class_distribution.png", dpi=150)
plt.show()
print(f"Imbalance ratio: {dist['class_imbalance']['imbalance_ratio']}")

In [ ]:
# Train/Val/Test counts for each setting
print("Split counts per setting:")
for name, n in CLASS_SETTINGS.items():
    splits, _, _ = load_split_data(n)
    print(f"  {name} (top-{n}): train={len(splits['train'][0])}, val={len(splits['val'][0])}, test={len(splits['test'][0])}")
    del splits
    gc.collect()

---
## Section 3 — Baseline Features

In [ ]:
print("Extracting baseline features (Top-20)...")
t0 = time.time()
feat_train = combined_baseline_features(X_train_20)
feat_val = combined_baseline_features(X_val_20)
feat_test = combined_baseline_features(X_test_20)
print(f"Done in {time.time()-t0:.1f}s")
print(f"Feature shape: {feat_train.shape}")

np.save(f"{OUT_DIR}/features/feat_train.npy", feat_train)
np.save(f"{OUT_DIR}/features/feat_val.npy", feat_val)
np.save(f"{OUT_DIR}/features/feat_test.npy", feat_test)
print(f"Saved to {OUT_DIR}/features/")

---
## Section 4 — OLMoEarth Embedding Extraction

In [ ]:
print("Loading OLMoEarth encoder...")
encoder = OLMoEarthEncoder(
    mode="local",
    local_weights_path=WEIGHTS_PATH,
    device=DEVICE
)
print(f"Encoder loaded (dim={encoder.model.dim})")

In [ ]:
print("Extracting embeddings (Top-20)...")
t0 = time.time()
emb_train = encoder.encode(X_train_20, batch_size=32)
emb_val = encoder.encode(X_val_20, batch_size=32)
emb_test = encoder.encode(X_test_20, batch_size=32)
print(f"Done in {time.time()-t0:.1f}s")
print(f"Embedding shape: {emb_train.shape}")

np.save(f"{OUT_DIR}/embeddings/emb_train.npy", emb_train)
np.save(f"{OUT_DIR}/embeddings/emb_val.npy", emb_val)
np.save(f"{OUT_DIR}/embeddings/emb_test.npy", emb_test)
np.save(f"{OUT_DIR}/embeddings/labels.npy", y_train_20)
print(f"Saved to {OUT_DIR}/embeddings/")

del X_train_20, X_val_20, X_test_20
gc.collect()

---
## Section 5 — Embedding Diagnostics

In [ ]:
print("Embedding Statistics (Train):")
print(f"  Shape:  {emb_train.shape}")
print(f"  Mean:   {emb_train.mean():.4f}")
print(f"  Std:    {emb_train.std():.4f}")
print(f"  Min:    {emb_train.min():.4f}")
print(f"  Max:    {emb_train.max():.4f}")
print(f"  Finite: {np.isfinite(emb_train).all()}")
print(f"  Unique (first 10): {np.unique(emb_train[:10], axis=0).shape[0]}/10")

In [ ]:
print("Generating UMAP...")
plot_umap(
    emb_train, y_train_20,
    title="OLMoEarth Embeddings - Top 20 Classes",
    save_path=f"{OUT_DIR}/figures/umap.png"
)
print(f"Saved to {OUT_DIR}/figures/umap.png")

---
## Section 6 — Classification Benchmark

In [ ]:
def run_experiment(name, X_tr, y_tr, X_te, y_te, clf_name, labels):
    """Run one experiment: scale -> train -> evaluate."""
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    clf = get_classifier(clf_name, seed=SEED)
    clf.fit(X_tr_s, y_tr)
    y_pred = clf.predict(X_te_s)
    m = compute_metrics(y_te, y_pred, labels=labels)
    m["method"] = name
    return m, y_pred

labels = sorted(set(y_test_20))
results = []
predictions = {}

In [ ]:
# Experiment A: Baseline Features + LogReg
print("[A] Baseline + LogReg...")
m, y_pred = run_experiment("Baseline + LogReg", feat_train, y_train_20, feat_test, y_test_20, "logreg", labels)
results.append(m)
predictions["Baseline + LogReg"] = y_pred
print(f"  OA={m['overall_accuracy']:.3f} F1={m['macro_f1']:.3f} Kappa={m['kappa']:.3f}")

# Experiment B: Baseline Features + LightGBM
print("[B] Baseline + LightGBM...")
m, y_pred = run_experiment("Baseline + LightGBM", feat_train, y_train_20, feat_test, y_test_20, "lgbm", labels)
results.append(m)
predictions["Baseline + LightGBM"] = y_pred
print(f"  OA={m['overall_accuracy']:.3f} F1={m['macro_f1']:.3f} Kappa={m['kappa']:.3f}")

# Experiment C: Embeddings + LogReg
print("[C] Embeddings + LogReg...")
m, y_pred = run_experiment("Embeddings + LogReg", emb_train, y_train_20, emb_test, y_test_20, "logreg", labels)
results.append(m)
predictions["Embeddings + LogReg"] = y_pred
print(f"  OA={m['overall_accuracy']:.3f} F1={m['macro_f1']:.3f} Kappa={m['kappa']:.3f}")

# Experiment D: Embeddings + LightGBM
print("[D] Embeddings + LightGBM...")
m, y_pred = run_experiment("Embeddings + LightGBM", emb_train, y_train_20, emb_test, y_test_20, "lgbm", labels)
results.append(m)
predictions["Embeddings + LightGBM"] = y_pred
print(f"  OA={m['overall_accuracy']:.3f} F1={m['macro_f1']:.3f} Kappa={m['kappa']:.3f}")

# Experiment E: Hybrid + LightGBM
print("[E] Hybrid + LightGBM...")
feat_train_all = np.concatenate([feat_train, emb_train], axis=1)
feat_test_all = np.concatenate([feat_test, emb_test], axis=1)
m, y_pred = run_experiment("Hybrid + LightGBM", feat_train_all, y_train_20, feat_test_all, y_test_20, "lgbm", labels)
results.append(m)
predictions["Hybrid + LightGBM"] = y_pred
print(f"  OA={m['overall_accuracy']:.3f} F1={m['macro_f1']:.3f} Kappa={m['kappa']:.3f}")

---
## Section 7 — Results Table

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df = df[["method", "overall_accuracy", "macro_f1", "weighted_f1", "kappa"]].copy()
df.columns = ["Method", "Accuracy", "Macro F1", "Weighted F1", "Kappa"]
df = df.sort_values("Macro F1", ascending=False).reset_index(drop=True)

print("=" * 70)
print(f"BENCHMARK RESULTS (Top-{PRIMARY_N} Classes)")
print("=" * 70)
print(df.to_string(index=False))
print("=" * 70)

df.to_csv(f"{OUT_DIR}/comparison_table.csv", index=False)
save_metrics(results, f"{OUT_DIR}/all_results.json")
print(f"Saved to {OUT_DIR}/")

---
## Section 8 — Confusion Matrix (Best Model)

In [ ]:
best_method = df.iloc[0]["Method"]
best_pred = predictions[best_method]
print(f"Best model: {best_method}")

cm = confusion_matrix(y_test_20, best_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            xticklabels=classes_20, yticklabels=classes_20)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — {best_method}')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/figures/confusion_matrix.png", dpi=150)
plt.show()
print(f"Saved to {OUT_DIR}/figures/confusion_matrix.png")

---
## Section 9 — Scaling Analysis

Run top experiments across Top-10, Top-20, Top-50 classes.

In [ ]:
def run_scaling_benchmark(top_n):
    """Run key experiments for a given number of classes."""
    splits, classes, le = load_split_data(top_n)
    X_tr, y_tr = splits["train"]
    X_te, y_te = splits["test"]
    labels = sorted(set(y_te))

    # Features
    feat_tr = combined_baseline_features(X_tr)
    feat_te = combined_baseline_features(X_te)

    # Embeddings
    enc = OLMoEarthEncoder(mode="local", local_weights_path=WEIGHTS_PATH, device=DEVICE)
    emb_tr = enc.encode(X_tr, batch_size=32)
    emb_te = enc.encode(X_te, batch_size=32)
    del enc; gc.collect()

    scaling_results = []
    for name, Xtr, Xte in [
        ("Baseline + LogReg", feat_tr, feat_te),
        ("Baseline + LightGBM", feat_tr, feat_te),
        ("Embeddings + LogReg", emb_tr, emb_te),
        ("Embeddings + LightGBM", emb_tr, emb_te),
        ("Hybrid + LightGBM", np.concatenate([feat_tr, emb_tr], axis=1),
                                np.concatenate([feat_te, emb_te], axis=1)),
    ]:
        clf_name = "logreg" if "LogReg" in name else "lgbm"
        m, _ = run_experiment(name, Xtr, y_tr, Xte, y_te, clf_name, labels)
        m["n_classes"] = top_n
        scaling_results.append(m)

    del splits, X_tr, X_te, feat_tr, feat_te, emb_tr, emb_te
    gc.collect()
    return scaling_results

In [ ]:
all_scaling = []
for n in [10, 20, 50]:
    print(f"\n--- Top-{n} Classes ---")
    t0 = time.time()
    res = run_scaling_benchmark(n)
    all_scaling.extend(res)
    print(f"Done in {time.time()-t0:.1f}s")

df_scale = pd.DataFrame(all_scaling)
df_scale = df_scale[["n_classes", "method", "overall_accuracy", "macro_f1", "kappa"]]
df_scale.columns = ["Classes", "Method", "Accuracy", "Macro F1", "Kappa"]
print("\n" + df_scale.to_string(index=False))
df_scale.to_csv(f"{OUT_DIR}/scaling_analysis.csv", index=False)

In [ ]:
# Scaling plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
methods = df_scale["Method"].unique()
for method in methods:
    sub = df_scale[df_scale["Method"] == method]
    axes[0].plot(sub["Classes"], sub["Accuracy"], "o-", label=method)
    axes[1].plot(sub["Classes"], sub["Macro F1"], "o-", label=method)
axes[0].set_title("Accuracy vs Number of Classes")
axes[0].set_xlabel("Top-N Classes")
axes[0].set_ylabel("Accuracy")
axes[0].legend(fontsize=7)
axes[1].set_title("Macro F1 vs Number of Classes")
axes[1].set_xlabel("Top-N Classes")
axes[1].set_ylabel("Macro F1")
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/figures/scaling_analysis.png", dpi=150)
plt.show()

---
## Section 10 — Final Interpretation

In [ ]:
best = df.iloc[0]
print("=" * 70)
print("FINAL INTERPRETATION")
print("=" * 70)
print(f"\nBest method: {best['Method']}")
print(f"  Accuracy:  {best['Accuracy']:.3f}")
print(f"  Macro F1:  {best['Macro F1']:.3f}")
print(f"  Kappa:     {best['Kappa']:.3f}")

baseline_best = df[df["Method"].str.startswith("Baseline")]["Macro F1"].max()
embed_best = df[df["Method"].str.startswith("Embeddings")]["Macro F1"].max()
hybrid_f1 = df[df["Method"].str.startswith("Hybrid")]["Macro F1"].max()

print(f"\n1. Do OLMoEarth embeddings outperform classical features?")
if embed_best > baseline_best:
    print(f"   YES — Embeddings ({embed_best:.3f}) > Baseline ({baseline_best:.3f})")
else:
    print(f"   NO  — Embeddings ({embed_best:.3f}) <= Baseline ({baseline_best:.3f})")

print(f"\n2. Does hybrid improve over both?")
if hybrid_f1 > max(baseline_best, embed_best):
    print(f"   YES — Hybrid ({hybrid_f1:.3f}) > max(Baseline, Embeddings) ({max(baseline_best, embed_best):.3f})")
else:
    print(f"   NO  — Hybrid ({hybrid_f1:.3f}) <= max ({max(baseline_best, embed_best):.3f})")

print(f"\n3. Scaling stability:")
if len(df_scale) > 0:
    for method in ["Baseline + LightGBM", "Embeddings + LightGBM", "Hybrid + LightGBM"]:
        sub = df_scale[df_scale["Method"] == method]
        if len(sub) > 1:
            f1_range = sub["Macro F1"].max() - sub["Macro F1"].min()
            print(f"   {method}: F1 range = {f1_range:.3f}")

print(f"\n4. Is OLMoEarth useful for EuroCropsML?")
if embed_best > 0.33 or hybrid_f1 > baseline_best:
    print("   YES — Embeddings provide useful representations")
else:
    print("   NOT YET — Need more data or better tuning")

print("=" * 70)

In [ ]:
print(f"Benchmark complete. All results saved to {OUT_DIR}/")